# Module 1 — Ship It: Your Assistant Is Alive

Welcome! Over 10 weeks you'll build a **personal AI assistant** that answers questions over a work-document library.

## 0. How to drive this notebook

A notebook is a stack of **cells**. Some cells are text (like this one); some are **code**.

- **To run a cell:** click it, then press **Shift + Enter** (or the ▶ button).
- **Run cells top-to-bottom.** Later cells use results from earlier ones.
- **If things break?** Worst case, use the menu: **Kernel → Restart & Run All** and
  the notebook returns to a clean state.

**If a cell shows an error:** Read the **last line only** — it names the problem. 

Try the first cell now.

In [ ]:
# This is a code cell. The '#' lines are comments — notes for humans, ignored by Python.
# print(...) shows text on the screen. The quotes hold the text.

print("Hello! I just ran my first cell.")

**That's a program.** You told the computer to do something and it did it.

### ✏️ Your turn #1 — edit and re-run

In the cell below, change the text **between the quotes** to anything you like (your name, your
job, your coffee order). Then run it. This edit-and-run loop is how the whole course works.

In [ ]:
my_message = "I work in ________ and I work as ________."

print(my_message)

## 1. Connect this notebook to the course code

Follow the [Week 1 setup](README.md#setup) from the repository root, then select
`rag-starter/.venv/bin/python` (Windows: `rag-starter/.venv/Scripts/python.exe`)
as this notebook's kernel. Keep `week-1/` and `rag-starter/` together in your checkout.

The cell below locates the course code. No API key is needed.


In [ ]:
import sys
from pathlib import Path

candidates = [Path.cwd(), Path.cwd() / "rag-starter"]
candidates.extend(base / "rag-starter" for base in Path.cwd().parents)
for candidate in candidates:
    if (candidate / "ribot").is_dir():
        RAG_STARTER = candidate.resolve()
        sys.path.insert(0, str(RAG_STARTER))
        break
else:
    raise FileNotFoundError("Open the notebook from the complete course checkout, alongside rag-starter.")
print(f"Connected to the course code at: {RAG_STARTER}")


## 2. Health check — is your machine ready?

One small check proves your setup works. **You only need to read three lines of the output:**

1. **Python:** your Python version (any 3.10+ is fine)
2. **LLM:** `offline (FakeLLM)` is **expected and fine** — see the box below
3. The final line: **All core checks passed. You're ready!**

> **What is FakeLLM?** The course ships a tiny *pretend* model so everything works **offline** —
> no account, no API key, no network, even on a locked-down work laptop. With a real key, the same code talks to a real model. Same assistant, two engines.

In [ ]:
# Keep every demo cell offline, even if this machine already has an API key.
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["RIBOT_LLM"] = ""
os.environ["RIBOT_EMBED"] = ""

import runpy
runpy.run_path(str(RAG_STARTER / "scripts" / "check_env.py"), run_name="__main__")
from ribot.chatbot import ChatBot


*(In a terminal, the same check is: `python scripts/check_env.py` from inside `rag-starter/` —
you'll do that in the assignment.)*

## 3. Meet your assistant

We'll have a short conversation with the assistant. Watch **what the second answer knows.**

In [ ]:
assistant = ChatBot()

print("you > My name is Sam.")
print("assistant >", assistant.reply("My name is Sam."))
print()
print("you > What's my name?")
print("assistant >", assistant.reply("What's my name?"))

**It remembered.** The second answer used information from the first turn. That is achieved via a Python list, and in Week 3 you will build it (and break it, and fix it) yourself.

### ✏️ Your turn #2 — make it yours

In the cell below, change the two messages: tell the assistant **your** name, then ask it
something. (Keep the "My name is ..." shape for the first one — the offline FakeLLM only knows
that one trick. A real model handles anything.)

In [ ]:
assistant = ChatBot()

print("assistant >", assistant.reply("My name is Alex."))   # <- put YOUR name here
print("assistant >", assistant.reply("What's my name?"))     # <- then try your own question

## 4. Look inside: the three parts

Every LLM assistant — including ChatGPT and the one you just ran — is built from **three parts**.
Each looks like magic today and becomes plain Python in a known week:

| # | Part | Plain English | You build it in |
|---|------|---------------|-----------------|
| 1 | **System prompt** | A paragraph of text that sets the assistant's job and personality | **Week 2** |
| 2 | **User message** | The text you send each turn | **Week 2** |
| 3 | **Memory** | A growing list that stores every turn of the conversation | **Week 3** (and Week 7) |

Let's see all three, live, in the assistant you just talked to:

In [ ]:
# GIVEN — run as-is and read the OUTPUT, not this code. The loop and the {:9s}
# formatting are Week 2-3 material — same deal as the plumbing cell above.
from ribot.chatbot import DEFAULT_SYSTEM

print("PART 1 — the system prompt is just text:")
print("   ", repr(DEFAULT_SYSTEM))
print()
print("PARTS 2 & 3 — the memory is a numbered list; your messages are in it:")
for i, m in enumerate(assistant.history):
    print(f"    [{i}] {m['role']:9s} : {m['content'][:70]}")

Read that output slowly, once:

- Line `[0]` is the **system prompt** (Part 1) — plain text.
- The lines marked `user` are **your messages** (Part 2).
- The whole numbered list **is the memory** (Part 3). Every turn adds two lines. That's the trick.

**Say it out loud in your own words** — "an assistant is: a personality paragraph, my message,
and a list that remembers the conversation." If you can say that, you have today's big idea.

> **Floor:** find the system prompt at `[0]`. **Stretch:** in the "your turn" cell above, add a
> third `assistant.reply(...)` line, predict how many rows the memory will grow by, then re-run
> both cells and check yourself. *(Answer: two — your message and its answer.)*

## 5. Two safety habits

1. **Secrets never go in code or Git.** API keys live in a `.env` file that Git is told to
   ignore. Never pasted into a notebook, script, or commit. (A leaked key = someone else's
   usage on your bill.)
2. **No real personal data.** Your assistant will index **synthetic or sanitized**
   documents only. Real customer/patient/employee data never enters the repo — and the same
   ban covers proprietary, client-confidential, and export-controlled (e.g., ITAR) material.
   Synthetic or sanitized documents count fully.


### Three ways to produce an answer

The Python around the model — prompts, memory, retrieval, and the interface — can stay the same while the **answer engine** changes:

| Mode | What produces the reply? | Internet/API key? | Why we use it |
|---|---|---|---|
| **Offline teaching mode** | `FakeLLM`, a predictable stand-in | No | Everyone can learn the pipeline today |
| **API model** | A model running on a provider's computers | Usually yes | Strong quality without running the model yourself |
| **Local model** | A small model running on your computer through a tool such as Ollama | No API key | Privacy, offline use, and control — with hardware and speed tradeoffs |

Today, just notice that these are **backends**, not three different assistants. There is nothing to install yet. In Week 2 we compare what users notice, Week 4 makes the swap a plain function boundary, Week 7 offers the first supported local run, and Week 10 turns the choice into an evidence-based decision.

> Important vocabulary: **running the app locally** means the Python program is on your machine. **Using a local model** means the model itself is also on your machine. Those are not automatically the same.

## 6. Glossary

You met real vocabulary today. Here's what each word means — no jargon:

| Word | Means |
|------|-------|
| **program** | Instructions a computer follows. You ran several today. |
| **Python** | The programming language this course uses. |
| **notebook / cell** | This document / one runnable box in it (Shift+Enter runs it). |
| **terminal** | A text window where you type commands to the computer (used in the assignment). |
| **repo (repository)** | A project folder tracked by **Git**, so every change is saved and shareable. Yours is `rag-starter/`. |
| **package** | A folder of reusable Python code. The course package is called `ribot`. |
| **API key** | A password that lets code use a paid AI model. Lives only in `.env`. |
| **offline mode / FakeLLM** | The course's pretend model — everything works with no key, no internet. |
| **local model** | A trained model that runs on your computer instead of behind an external API. |
| **Ollama** | A tool that downloads and serves compatible models locally; it is a runner, not the model itself. |
| **system prompt** | The text paragraph that gives the assistant its job and personality. |

For reference only. No need to memorize this. These words will show up again later.